# CIFAR-10 — Exploratory Data Analysis

This notebook explores the CIFAR-10 dataset before training. Use it to:
- Understand class distribution
- Visualize augmentation effects
- Inspect pixel statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator

CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 1. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = np.bincount(y_train.flatten())
axes[0].bar(CLASS_NAMES, counts, color=sns.color_palette('husl', 10))
axes[0].set_title('Training Set — Class Distribution')
axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
axes[0].set_ylabel('Count')

counts_test = np.bincount(y_test.flatten())
axes[1].bar(CLASS_NAMES, counts_test, color=sns.color_palette('husl', 10))
axes[1].set_title('Test Set — Class Distribution')
axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha='right')
plt.tight_layout()
plt.show()
print(f'Each class has exactly {counts[0]} training samples — perfectly balanced!')

## 2. Sample Images per Class

In [ ]:
fig, axes = plt.subplots(10, 8, figsize=(14, 18))
fig.suptitle('8 Samples per Class', fontsize=14)
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    idxs = np.where(y_train.flatten() == cls_idx)[0][:8]
    for col, idx in enumerate(idxs):
        axes[cls_idx][col].imshow(X_train[idx])
        axes[cls_idx][col].axis('off')
        if col == 0:
            axes[cls_idx][col].set_ylabel(cls_name, fontsize=9, rotation=0,
                                          labelpad=40, va='center')
plt.tight_layout()
plt.show()

## 3. Pixel Statistics

In [ ]:
X_norm = X_train.astype('float32') / 255.0
print('Channel means (R, G, B):', X_norm.mean(axis=(0,1,2)))
print('Channel stds  (R, G, B):', X_norm.std(axis=(0,1,2)))

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
colors = ['red', 'green', 'blue']
for i, (ax, color) in enumerate(zip(axes, colors)):
    ax.hist(X_norm[:1000, :, :, i].flatten(), bins=50, color=color, alpha=0.7)
    ax.set_title(f'{color.upper()} channel distribution')
    ax.set_xlabel('Pixel value')
plt.tight_layout()
plt.show()

## 4. Data Augmentation Preview

In [ ]:
sample = X_norm[0:1]
datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1,
                              height_shift_range=0.1, horizontal_flip=True,
                              zoom_range=0.1)

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
fig.suptitle('Original (top row) vs Augmented (bottom row)', fontsize=12)
for i in range(8):
    axes[0][i].imshow(X_norm[i])
    axes[0][i].axis('off')

gen = datagen.flow(X_norm[:8], batch_size=8)
aug_batch = next(gen)
for i in range(8):
    axes[1][i].imshow(np.clip(aug_batch[i], 0, 1))
    axes[1][i].axis('off')
plt.tight_layout()
plt.show()